In [1]:
!pip install fastapi nest-asyncio pyngrok uvicorn

In [2]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 123.3 MB/s eta 0:00:00


In [3]:
#### This is used for T4 GPU

!uv pip install vllm

Using Python 3.13.15 environment at: /usr
Resolved 198 packages in 1.13s
Prepared 84 packages in 58.04s
Uninstalled 16 packages in 1.13s
Installed 84 packages in 281ms
 + agent-detector==2.0.0
 + anthropic==1.11.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.10
 + cbor2==6.1.5
 + compressed-tensors==0.17.0
 - cuda-core==0.7.0
 + cuda-core==1.2.1
 + cuda-tile==1.6.0
 - cuda-toolkit==13.0.2
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + fastsafetensors==0.4.0
 + flashinfer-python==0.7.0.post1
 + humming-kernels==0.1.16
 + ijson==3.5.1
 + instanttensor==0.2.1
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.44.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mcp==2.3.0
 + mcp-types==2.3.0
 + mistral-common==1.12.0
 + m

In [ ]:
#### This is used for T4 GPU

# !uv pip uninstall vllm torch torchvision torchaudio
# !uv pip install torch==<version> --index-url https://download.pytorch.org/whl/cu129
# !uv pip install vllm --extra-index-url https://wheels.vllm.ai/0.19.0/cu129 --extra-index-url https://download.pytorch.org/whl/cu129

In [ ]:
# !ln -s /usr/local/cuda/lib64/libcudart.so.12 /usr/local/cuda/lib64/libcudart.so.13

In [4]:
!vllm -h

usage: vllm [-h] [-v]
            {chat,complete,serve,launch,bench,collect-env,preload,run-batch,snapshot} ...

vLLM CLI

positional arguments:
  {chat,complete,serve,launch,bench,collect-env,preload,run-batch,snapshot}
    chat                Generate chat completions via the running API server.
    complete            Generate text completions based on the given prompt
                        via the running API server.
    serve               Launch a local OpenAI-compatible API server to serve
                        LLM completions via HTTP.
    launch              Launch individual vLLM components.
    bench               vLLM bench subcommand.
    collect-env         Start collecting environment information.
    preload             Launch weight cache daemons (one per GPU) for fast
                        engine restarts.
    run-batch           Run batch prompts and write results to file.
    snapshot            Create, inspect, or restore an initialized vLLM
                 

In [95]:
import subprocess

log_file = open("vllm.log", "w")

proc = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen3.5-4B",
        "--trust-remote-code",
        "--tensor-parallel-size", "1",
        "--reasoning-parser", "qwen3",
        "--default-chat-template-kwargs", '{"enable_thinking": false}',
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":1}',
        "--language-model-only",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Started, PID:", proc.pid)

Started, PID: 45909


In [100]:
pid = proc.pid
pid

45909

In [101]:
# !kill {pid}

In [93]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3.5-9B","object":"model","created":1791336397,"owned_by":"vllm","root":"Qwen/Qwen3.5-9B","parent":null,"max_model_len":262144,"permission":[{"id":"modelperm-b4648ad0b0a03fb9","object":"model_permission","created":1791336397,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [90]:
# !vllm serve Qwen/Qwen3.5-9B \
#   --trust-remote-code \
#   --tensor-parallel-size 1 \
#   --enable-auto-tool-choice \
#   --tool-call-parser qwen3_coder \
#   --speculative-config '{"method":"mtp","num_speculative_tokens":1}' \
#   --language-model-only

(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347] 
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen3.5-9B
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347] 
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen3.5-9B', 'enable_auto_tool_choice': True, 'tool_call_parser': 'qwen3_coder', 'model': 'Qwen/Qwen3.5-9B', 'trust_remote_code': True, 'language_model_only': True, 'speculative_config': {'method': 'mtp', 'num_speculative_tokens': 1}}
(APIServer pid=32959) INFO 10-07 00:40:48 [model.py:699] Resolved architecture: Qwen3_5ForConditionalGeneration
(APIServer pid=32959) INFO 10-07 00:4

In [ ]:
# !vllm serve {MODEL_NAME} --port 8000 \
#     --gpu-memory-utilization 0.90 --max-model-len 2048\
#     --dtype half --enable-auto-tool-choice \
#     --tool-call-parser qwen3_coder \
#     --language-model-only \
#     --enable-prefix-caching \
#     --default-chat-template-kwargs '{"enable_thinking": false}

In [14]:
!git clone https://github.com/adelelwan24/Story-Retrieval-Chatbot.git

Cloning into 'Story-Retrieval-Chatbot'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 90 (delta 15), reused 89 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 34.99 MiB | 31.82 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [15]:
%cd Story-Retrieval-Chatbot/

/content/Story-Retrieval-Chatbot


In [54]:
!pwd

/content/Story-Retrieval-Chatbot


In [19]:
!cat /content/Story-Retrieval-Chatbot/configs/chat.yaml


# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  server_model: Qwen/Qwen3.5-9B     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut be

In [97]:
%%writefile /content/Story-Retrieval-Chatbot/configs/chat.yaml

# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  # server_model: Qwen/Qwen3.5-9B     # model name the server expects
  server_model: Qwen/Qwen3.5-4B     # model name the server expects
  # server_model: google/gemma-4-E2B-it     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut before the LLM sees them
  history_turns: 4                     # past user/answer pairs kept for follow-ups ("summarize it")

tools:
  title_threshold: 85                  # fuzzy title score (0-100) needed to accept a title match
  genre_threshold: 80                  # fuzzy genre score needed to use the genre as a filter
  dense_vector: dense_late             # dense_late (late chunking) or dense_prefix (baseline)
  prefetch_limit: 100                  # candidates per retriever (dense, BM25) before RRF fusion
  default_num_stories: 6               # theme searches return at least this many different stories
  max_num_stories: 20
  passages_per_story: 2                # best chunks shown per story in search results
  snippet_chars: 300                   # passage preview length in search results
  num_passages: 4                      # passages returned by ask_about_story for long stories
  full_story_chars: 6000               # ask_about_story gives stories up to this size whole
  max_story_chars: 12000               # cap on story text inside one observation
  summary_chunk_chars: 10000           # longer stories are summarised part by part, then merged
  list_limit: 50

Overwriting /content/Story-Retrieval-Chatbot/configs/chat.yaml


In [21]:
!pip install -r requirements.txt

Obtaining file:///content/Story-Retrieval-Chatbot (from -r requirements.txt (line 2))
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 37.2 MB/s eta 0:00:00
  Building editable for storybot (pyproject.toml) ... done
  Created wheel for storybot: filename=storybot-0.1.0-0.editable-py3-none-any.whl size=1425 sha256=9cd6823b995db9b9673c8d6996328bd13b8fecede841b4011320f401390175ed
  Sto

In [99]:
!python /content/Story-Retrieval-Chatbot/scripts/chat.py --trace

Loading weights: 100% 134/134 [00:00<00:00, 5554.99it/s]

you> In The Nightmare of Neon Valley, who is Victor?
[1] Action: get_story_by_title({"title": "The Nightmare of Neon Valley"})
[1] Observation: {"found": true, "matched_title": "The Nightmare of Neon Valley", "match_score": 100.0, "story": {"story_id": 794662, "title": "The Nightmare of Neon Valley", "genre": "Crime", "text": "In the heart of a bustling city called Neon Valley, a crime wave had begun to sweep through the streets. At the for...

In "The Nightmare of Neon Valley" (id 794662), Victor is a former henchman of Isabella (the leader of the Scarlet Syndicate) who turns against her. He witnesses the devastation Isabella has caused and decides to help Jack Harper bring her down. Victor becomes Jack's unexpected ally in the final battle against Isabella and the Scarlet Syndicate.

you> Which genres do you have?
[1] Action: list_genres({})
[1] Observation: {"num_genres": 99, "genres": ["Adventure", "Alternate Dimension", "Al

## Test `google/gemma-4-E2B-it`

In [104]:
!wget -q -O tool_chat_template_gemma4.jinja \
  https://raw.githubusercontent.com/vllm-project/vllm/main/examples/tool_chat_template_gemma4.jinja
!ls -l tool_chat_template_gemma4.jinja

-rw-r--r-- 1 root root 18653 Oct  7 02:06 tool_chat_template_gemma4.jinja


In [127]:
!vllm serve --help=all | grep -i -A3 "reasoning-parser"

  --reasoning-parser REASONING_PARSER
                        Select the reasoning parser depending on the model
                        that you're using. This is used to parse the reasoning
                        content into OpenAI API format. (default: )
  --reasoning-parser-plugin REASONING_PARSER_PLUGIN
                        Path to a dynamically reasoning parser plugin that can
                        be dynamically loaded and registered. (default: )



In [129]:
import subprocess

log_file = open("vllm.log", "w")

proc = subprocess.Popen(
    [
        "vllm", "serve", "google/gemma-4-E2B-it",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "gemma4",
        "--reasoning-parser", "gemma4",
        "--chat-template", "tool_chat_template_gemma4.jinja",
        "--language-model-only",
        "--speculative-config",
        '{"model":"google/gemma-4-E2B-it-assistant","num_speculative_tokens":2}',
        "--default-chat-template-kwargs", '{"enable_thinking": false}',
        # "--kv-cache-dtype", "fp8",
        "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',
        "--async-scheduling",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Started, PID:", proc.pid)

Started, PID: 67298


In [128]:
import os, signal
try:
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
except Exception as e:
    print("Nothing to stop:", e)

In [131]:
!tail -n 50 vllm.log

Capturing CUDA graphs (FULL): 100%|██████████| 56/56 [00:03<00:00, 18.18it/s]
(EngineCore pid=67493) INFO 10-07 02:45:25 [speculator.py:144] Capturing model for speculator...
Capturing decode CUDA graphs (FULL): 100%|██████████| 38/38 [00:00<00:00, 102.19it/s]
(EngineCore pid=67493) INFO 10-07 02:45:27 [model_runner.py:1087] Graph capturing finished in 10 secs, took 1.23 GiB
(EngineCore pid=67493) INFO 10-07 02:45:27 [gpu_worker.py:892] CUDA graph pool memory: 1.23 GiB (actual), 1.65 GiB (estimated), difference: 0.42 GiB (34.4%).
(EngineCore pid=67493) INFO 10-07 02:45:27 [gpu_worker.py:955] Free memory on device (39.08/39.49 GiB) on startup. Desired GPU memory utilization is (0.92, 36.33 GiB). Actual usage is 9.47 GiB for consumed memory (weights + non-torch), 2.17 GiB for peak activation, and 1.23 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=25029353534` (23.31 GiB) to fit into requested memory, or `--kv-cache-memory=27973096960` (26.05 GiB)

In [111]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"google/gemma-4-E2B-it","object":"model","created":1791339361,"owned_by":"vllm","root":"google/gemma-4-E2B-it","parent":null,"max_model_len":131072,"permission":[{"id":"modelperm-8fda3915491124bb","object":"model_permission","created":1791339361,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [113]:
# @title set-chat-config
%%writefile /content/Story-Retrieval-Chatbot/configs/chat.yaml

# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  # server_model: Qwen/Qwen3.5-9B     # model name the server expects
  # server_model: Qwen/Qwen3.5-4B     # model name the server expects
  server_model: google/gemma-4-E2B-it     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut before the LLM sees them
  history_turns: 4                     # past user/answer pairs kept for follow-ups ("summarize it")

tools:
  title_threshold: 85                  # fuzzy title score (0-100) needed to accept a title match
  genre_threshold: 80                  # fuzzy genre score needed to use the genre as a filter
  dense_vector: dense_late             # dense_late (late chunking) or dense_prefix (baseline)
  prefetch_limit: 100                  # candidates per retriever (dense, BM25) before RRF fusion
  default_num_stories: 6               # theme searches return at least this many different stories
  max_num_stories: 20
  passages_per_story: 2                # best chunks shown per story in search results
  snippet_chars: 300                   # passage preview length in search results
  num_passages: 4                      # passages returned by ask_about_story for long stories
  full_story_chars: 6000               # ask_about_story gives stories up to this size whole
  max_story_chars: 12000               # cap on story text inside one observation
  summary_chunk_chars: 10000           # longer stories are summarised part by part, then merged
  list_limit: 50

Overwriting /content/Story-Retrieval-Chatbot/configs/chat.yaml


In [132]:
!python /content/Story-Retrieval-Chatbot/scripts/chat.py --trace

Loading weights: 100% 134/134 [00:00<00:00, 4699.15it/s]

you> Who leads the Resurgence in Harmony Chronicles Unveiled?
[1] Action: get_story_by_title({"title": "Harmony Chronicles Unveiled"})
[1] Observation: {"found": true, "matched_title": "Harmony Chronicles Unveiled", "match_score": 100.0, "story": {"story_id": 468507, "title": "Harmony Chronicles Unveiled", "genre": "Utopian", "text": "In the year 2095, the world had undergone a remarkable transformation. After centuries of conflict and strife, huma...

The Resurgence in "Harmony Chronicles Unveiled" is led by Malcolm Vore.

you> In The Chronicles of the Forgotten Kingdom, what monster does King Harishchandra fight on the way to the mountain?
[1] Action: get_story_by_title({"title": "The Chronicles of the Forgotten Kingdom"})
[1] Observation: {"found": true, "matched_title": "The Chronicles of the Forgotten Kingdom", "match_score": 100.0, "story": {"story_id": 983437, "title": "The Chronicles of the Forgotten Kingdom", "genre": "

In [ ]:
import requests

def check_vllm_status():
    try:
        response = requests.get("http://localhost:8000/health")
        if response.status_code == 200:
            print("vllm server is running")
            return True
    except requests.exceptions.ConnectionError:
        print("vllm server is not running")
        return False

try:
    # Monitor the process
    while True:
        if check_vllm_status() == True:
            print("The vllm server is ready to serve.")
            break
        else:
            print("The vllm server has stopped.")
            stdout, stderr = proc.communicate(timeout=10)
            print(f"STDOUT: {stdout.decode('utf-8')}")
            print(f"STDERR: {stderr.decode('utf-8')}")
            break
        time.sleep(5)  # Check every second
except KeyboardInterrupt:
    print("Stopping the check of vllm...")